In [200]:
using LowLevelFEM

In [201]:
openGeometry("node-to-segment.geo")
#openPreProcessor()

In [202]:
mat_seg = Material("segment")
mat_nod = Material("node")

U = Field([mat_seg, mat_nod], type=:VectorField, dim=2, fieldName=:u)

Problem("node-to-segment", :VectorField, 2, 2, Material[Material("segment", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("node", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 4, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs, false)

In [203]:
bc1 = BoundaryCondition("left", ux=0, uy=0)
bc2 = BoundaryCondition("right", ux=0, uy=0)
bc3 = BoundaryCondition("node", ux=0, uy=0)

BoundaryCondition("node", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0, :ux => 0))

In [204]:
r = nodePositionVector(U)
r = projectTo2D(r)
DoFs(r)

8×1 Matrix{Float64}:
 0.0
 0.0
 1.0
 1.0
 0.0
 1.0
 0.0001414213562373095
 1.0001414213562374

In [205]:
u = applyBoundaryConditions(U, [bc1, bc2, bc3])
DoFs(u)

8×1 Matrix{Float64}:
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0

In [206]:
DoFs(r+u)

8×1 Matrix{Float64}:
 0.0
 0.0
 1.0
 1.0
 0.0
 1.0
 0.0001414213562373095
 1.0001414213562374

In [207]:
C = contact(U, master="segment", slave="node")

Contact("node" -> "segment", 2 candidate nodes, 0 active, G=(4, 8), Pa=(0, 4))

In [208]:
C.G[:,:]

4×8 SparseArrays.SparseMatrixCSC{Float64, Int64} with 24 stored entries:
  0.353553  -0.353553   0.353553  -0.353553  …  0.707107    ⋅         ⋅ 
 -0.353553  -0.353553  -0.353553  -0.353553     0.707107    ⋅         ⋅ 
  0.353453  -0.353453   0.353653  -0.353653      ⋅        -0.707107  0.707107
 -0.353453  -0.353453  -0.353653  -0.353653      ⋅         0.707107  0.707107

In [209]:
C.G.A * r.a

4×1 Matrix{Float64}:
 0.7071067811865476
 1.1102230246251565e-16
 0.7071067811865477
 7.739489225016005e-17

In [210]:
bc4 = BoundaryCondition("node", ux=1.1, uy=-0.9)

u = applyBoundaryConditions(U, [bc1, bc2, bc4])
updateContact!(C, u)

Contact("node" -> "segment", 2 candidate nodes, 2 active, G=(4, 8), Pa=(4, 4))

In [211]:
C.G[:,:]

4×8 SparseArrays.SparseMatrixCSC{Float64, Int64} with 24 stored entries:
  0.282843  -0.282843   0.424264  -0.424264  …  0.707107    ⋅         ⋅ 
 -0.282843  -0.282843  -0.424264  -0.424264     0.707107    ⋅         ⋅ 
  0.282743  -0.282743   0.424364  -0.424364      ⋅        -0.707107  0.707107
 -0.282743  -0.282743  -0.424364  -0.424364      ⋅         0.707107  0.707107

In [212]:
C.G.A * (r.a + u.a)

4×1 Matrix{Float64}:
 -0.7071067811865477
  2.8903078423695096e-15
 -0.7071067811865477
  2.7179025540790094e-15

In [213]:
Cn = ∫(U ⋅ U, Γ="node")
Cn[:,:]

8×8 SparseArrays.SparseMatrixCSC{Float64, Int64} with 8 stored entries:
  ⋅    ⋅    ⋅    ⋅    ⋅           ⋅           ⋅           ⋅ 
  ⋅    ⋅    ⋅    ⋅    ⋅           ⋅           ⋅           ⋅ 
  ⋅    ⋅    ⋅    ⋅    ⋅           ⋅           ⋅           ⋅ 
  ⋅    ⋅    ⋅    ⋅    ⋅           ⋅           ⋅           ⋅ 
  ⋅    ⋅    ⋅    ⋅   6.66667e-5   ⋅          3.33333e-5   ⋅ 
  ⋅    ⋅    ⋅    ⋅    ⋅          6.66667e-5   ⋅          3.33333e-5
  ⋅    ⋅    ⋅    ⋅   3.33333e-5   ⋅          6.66667e-5   ⋅ 
  ⋅    ⋅    ⋅    ⋅    ⋅          3.33333e-5   ⋅          6.66667e-5

In [214]:
cn = 1
ct = 1

Dc = [cn 0
      0  ct]

C0 = ∫(U ⋅ Dc ⋅ U; Γ="node")

CC = subSystemMatrix(
    C0;
    onPhysicalGroup="node"
)

CC[:,:]

4×4 SparseArrays.SparseMatrixCSC{Float64, Int64} with 8 stored entries:
 6.66667e-5   ⋅          3.33333e-5   ⋅ 
  ⋅          6.66667e-5   ⋅          3.33333e-5
 3.33333e-5   ⋅          6.66667e-5   ⋅ 
  ⋅          3.33333e-5   ⋅          6.66667e-5

In [215]:
Ga = C.Pa * C.G
Ca = C.Pa * CC * C.Pa'

Kc = Ga' * Ca * Ga

Kc[:,:]

8×8 SparseArrays.SparseMatrixCSC{Float64, Int64} with 64 stored entries:
  3.19887e-5   0.0          4.79972e-5  …  -3.99906e-5   0.0
  0.0          3.19887e-5   0.0             0.0         -3.99906e-5
  4.79972e-5   0.0          7.2017e-5      -6.00094e-5   0.0
  0.0          4.79972e-5   0.0             0.0         -6.00094e-5
 -3.99953e-5   0.0         -6.00047e-5      3.33333e-5   0.0
  0.0         -3.99953e-5   0.0         …   0.0          3.33333e-5
 -3.99906e-5   0.0         -6.00094e-5      6.66667e-5   0.0
  0.0         -3.99906e-5   0.0             0.0          6.66667e-5

In [216]:
C.Pa[:,:]

4×4 SparseArrays.SparseMatrixCSC{Float64, Int64} with 4 stored entries:
 1.0   ⋅    ⋅    ⋅ 
  ⋅   1.0   ⋅    ⋅ 
  ⋅    ⋅   1.0   ⋅ 
  ⋅    ⋅    ⋅   1.0

In [217]:
C.G.A * (r.a + u.a)

4×1 Matrix{Float64}:
 -0.7071067811865477
  2.8903078423695096e-15
 -0.7071067811865477
  2.7179025540790094e-15